# Módulo 21: Práctica de Prompt Engineering & Context Engineering
**Diplomado en Inteligencia Artificial y Salud 2026**

Este Jupyter Notebook complementa la **Presentación 21**. Aquí ejecutaremos ejemplos prácticos en Python para dominar:
1. **Anatomía de Prompts** y Delimitadores XML/Markdown.
2. **Zero-Shot vs Few-Shot Prompting**.
3. **Salidas Estructuradas con Pydantic**.
4. **Chain-of-Thought (CoT)** para razonamiento clínico y matemático.
5. **Patrón ReAct (Reasoning + Acting)** en un bucle interactivo.
6. **Context Engineering**: Gestión de Memoria y Scratchpads.

## 1. Configuración de Entorno e Importaciones
Importamos las librerías necesarias (Pydantic para validación y tipos de datos).

In [ ]:
import json
from typing import List, Optional
from pydantic import BaseModel, Field

print("✅ Entorno preparado para Prompt & Context Engineering")

## 2. Ejemplo 1: Plantilla de Prompt Delimitada con XML Tags
Demostración de cómo construir un prompt robusto que separa instrucciones, contexto y datos.

In [ ]:
def construir_prompt_clinico(system_role: str, instrucciones: str, reporte: str) -> str:
    prompt_template = f"""<system_role>
{system_role}
</system_role>

<instructions>
{instrucciones}
</instructions>

<patient_data>
{reporte}
</patient_data>

<output_format>
Responde únicamente en formato JSON con la clave 'diagnosticos_probables' y 'recomendacion'.
</output_format>"""
    return prompt_template

system_role = "Eres un asistente médico experto en medicina interna."
instrucciones = "Analiza la sintomatología y extrae los 2 diagnósticos más probables."
reporte = "Paciente masculino de 58 años presenta disnea progresiva, tos seca y astenia de 3 semanas."

prompt_final = construir_prompt_clinico(system_role, instrucciones, reporte)
print(prompt_final)

## 3. Ejemplo 2: Salidas Estructuradas con Pydantic
Definición de esquemas rigurosos para extraer entidades médicas sin errores de formato.

In [ ]:
class ExtraccionDiagnostico(BaseModel):
    paciente_id: str = Field(description="Identificador único del paciente")
    sintomas: List[str] = Field(description="Lista de síntomas identificados")
    nivel_severidad: str = Field(description="Leve, Moderado o Grave")
    requiere_hospitalizacion: bool

# Simulación de respuesta estructurada validada
ejemplo_json_raw = '{"paciente_id": "PAC-2026-99", "sintomas": ["disnea", "tos seca"], "nivel_severidad": "Moderado", "requiere_hospitalizacion": false}'
objeto_validado = ExtraccionDiagnostico.model_validate_json(ejemplo_json_raw)

print("Objeto Pydantic Validado con éxito:")
print(objeto_validado)
print(f"Nivel de Severidad: {objeto_validado.nivel_severidad}")

## 4. Ejemplo 3: Patrón ReAct (Reasoning + Acting)
Implementación simplificada en Python de un bucle de agente ReAct que consulta herramientas externas.

In [ ]:
def herramienta_busqueda_farmacos(farmaco: str) -> str:
    """Simulación de base de datos farmacológica."""
    db = {
        "paracetamol": "Dosis máx: 4g/día en adultos. En falla hepática máx 2g/día.",
        "ibuprofeno": "Dosis máx: 1200mg/día sin prescripción. Precaución en úlcera péptica."
    }
    return db.get(farmaco.lower(), "Farmaco no encontrado en la base de datos.")

class AgenteReActSimulado:
    def __init__(self):
        self.scratchpad = []
        
    def ejecutar_paso(self, pregunta: str):
        print(f"❓ Pregunta: {pregunta}\n")
        
        # Paso 1: Thought
        thought1 = "Thought: Debo consultar la dosis máxima de Paracetamol en la base de fármacos."
        self.scratchpad.append(thought1)
        print(thought1)
        
        # Paso 2: Action
        action1 = "Action: herramienta_busqueda_farmacos('Paracetamol')"
        self.scratchpad.append(action1)
        print(action1)
        
        # Paso 3: Observation
        obs1 = f"Observation: {herramienta_busqueda_farmacos('paracetamol')}"
        self.scratchpad.append(obs1)
        print(obs1)
        
        # Paso 4: Final Answer
        final_ans = "Final Answer: La dosis máxima de Paracetamol es 4g/día en adultos y 2g/día en pacientes con falla hepática."
        print(f"\n🏆 {final_ans}")

agente = AgenteReActSimulado()
agente.ejecutar_paso("¿Cuál es la dosis máxima segura de Paracetamol?")

## 5. Ejemplo 4: Context Engineering - Gestión de Scratchpad y Memoria
Demostración de cómo se mantiene actualizado el estado del contexto de un agente.

In [ ]:
class GestorContextoAgente:
    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        self.scratchpad_notas = []
        self.documentos_rag = []
        
    def agregar_nota_scratchpad(self, nota: str):
        self.scratchpad_notas.append(nota)
        
    def agregar_fragmento_rag(self, fragmento: str):
        self.documentos_rag.append(fragmento)
        
    def ensamblar_contexto_eficiente(self, prompt_usuario: str) -> str:
        contexto_ensamblado = f"""<system_prompt>
{self.system_prompt}
</system_prompt>

<agent_scratchpad_memory>
""".join(f"- {n}" for n in self.scratchpad_notas)}
</agent_scratchpad_memory>

<rag_context>
""".join(f"[Doc]: {d}" for d in self.documentos_rag)}
</rag_context>

<user_query>
{prompt_usuario}
</user_query>"""
        return contexto_ensamblado

gestor = GestorContextoAgente("Eres un agente de Context Engineering para el diplomado.")
gestor.agregar_nota_scratchpad("Paso 1 completado: Extracción de síntomas realizada con éxito.")
gestor.agregar_fragmento_rag("Guía de Práctica Clínica: Evaluación de Cefaleas (2025).")

contexto_completo = gestor.ensamblar_contexto_eficiente("Genera el informe final del paciente.")
print(contexto_completo)